#### Create StorageClass

In [ ]:
cat > storageclass.yaml << OF
apiVersion: storage.k8s.io/v1
kind: StorageClass
metadata:
    name: local-storage
provisioner: kubernetes.io/no-provisioner
volumeBindingMode: WaitForFirstConsumer
OF

kubectl apply -f storageclass.yaml

#### master-1 ( nexus )

In [ ]:
sudo mkdir -p /data/nexus
sudo chmod 777 /data/nexus

In [ ]:
cat > pv-nexus-master1.yaml <<EOF
apiVersion: v1
kind: PersistentVolume
metadata:
  name: pv-nexus-master1
spec:
  capacity:
    storage: 20Gi
  accessModes:
    - ReadWriteOnce
  storageClassName: local-storage
  persistentVolumeReclaimPolicy: Retain
  local:
    path: /data/nexus
  nodeAffinity:
    required:
      nodeSelectorTerms:
      - matchExpressions:
        - key: kubernetes.io/hostname
          operator: In
          values:
            - master-1
EOF

kubectl apply -f pv-nexus-master1.yaml

Verify

In [ ]:
kubectl get pv

PVC must be in the same name space with app

In [ ]:
kubectl create namespace nexus

In [ ]:
cat > pvc-nexus.yaml <<EOF
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: nexus-pvc
  namespace: nexus

spec:
  accessModes:
    - ReadWriteOnce

  resources:
    requests:
      storage: 20Gi

  storageClassName: local-storage

  volumeName: pv-nexus-master1
EOF

kubectl apply -f pvc-nexus.yaml

Add Helm Repo

In [ ]:
helm repo add sonatype https://sonatype.github.io/helm3-charts/
helm repo update

In [ ]:
cat > nexus-values.yaml <<EOF
persistence:
  enabled: true
  existingClaim: nexus-pvc

tolerations:
- key: "node-role.kubernetes.io/control-plane"
  operator: "Exists"
  effect: "NoSchedule"

nodeSelector:
  kubernetes.io/hostname: master-1

ingress:
  enabled: true
  ingressClassName: nginx
  hostPath: /
  hostRepo: nexus.voip.local
  annotations:
    nginx.ingress.kubernetes.io/proxy-body-size: "0"
    nginx.ingress.kubernetes.io/proxy-read-timeout: "600"
    nginx.ingress.kubernetes.io/proxy-send-timeout: "600"

nexus:
  nexusPort: 8081   # ← make sure ingress targets this port

readinessProbe:
  initialDelaySeconds: 120    # give Nexus 2 min before first check
  periodSeconds: 15
  failureThreshold: 6
  httpGet:
    path: /service/rest/v1/status
    port: 8081

livenessProbe:
  initialDelaySeconds: 120
  periodSeconds: 15
  failureThreshold: 6
  httpGet:
    path: /service/rest/v1/status
    port: 8081
env:
  - name: INSTALL4J_ADD_VM_PARAMS
    value: "-Xms2703M -Xmx2703M -XX:MaxDirectMemorySize=2703M -XX:+UnlockExperimentalVMOptions -XX:+UseCGroupMemoryLimitForHeap -Djava.util.prefs.userRoot=/nexus-data/javaprefs"
EOF


helm install nexus sonatype/nexus-repository-manager \
  -n nexus \
  -f nexus-values.yaml

In [ ]:
kubectl exec -n nexus -it deployment/nexus-nexus-repository-manager -- cat /nexus-data/admin.password

---

In [ ]:
cat > nexus-docker-service.yaml << 'EOF'
apiVersion: v1
kind: Service
metadata:
  name: nexus-docker-registry
  namespace: nexus
  labels:
    app.kubernetes.io/name: nexus-repository-manager
    app.kubernetes.io/instance: nexus
spec:
  type: NodePort
  selector:
    app.kubernetes.io/name: nexus-repository-manager
    app.kubernetes.io/instance: nexus
  ports:
    - name: docker-hosted
      port: 8086
      targetPort: 8086
      nodePort: 30086
EOF
kubectl apply -f nexus-docker-service.yaml

Verify

In [ ]:
kubectl get svc -n nexus

---

Add repo in nexus

In [ ]:
How to Create Them

    Create Blob Store (for Docker cache):
        Admin → Repository → Blob Stores → Create blob store
        Name: docker-cache
        Type: File
        Path: /nexus-data/blobs/docker-cache (or use default)

    Repository → Create repository → docker (hosted)
    Name: docker-hosted
    HTTP: ✅ Enable, Port: 8086
    Allow anonymous docker pull: ✅
    Blob store: default (or docker-blob)

    Enable Docker Bearer Token Realm:
        Security → Realms → Move Docker Bearer Token Realm to Active

---

- On every nodes

In [ ]:
for REG in docker.io registry.k8s.io ghcr.io quay.io; do
  sudo mkdir -p /etc/containerd/certs.d/$REG
  sudo tee /etc/containerd/certs.d/$REG/hosts.toml << EOF
server = "https://$REG"
[host."http://master-1:30086"]
  capabilities = ["pull", "resolve"]
  skip_verify = true
EOF
done

mkdir -p /etc/containerd/certs.d/master-1:30086

cat > /etc/containerd/certs.d/master-1:30086/hosts.toml <<EOF
server = "http://master-1:30086"

[host."http://master-1:30086"]
  capabilities = ["pull", "resolve", "push"]
  skip_verify = true
EOF



sudo systemctl restart containerd


In [ ]:
# Create the override file
mkdir -p /etc/containerd/conf.d

cat > /etc/containerd/conf.d/registry-mirrors.toml << 'EOF'
[plugins.'io.containerd.cri.v1.images'.registry]
  config_path = '/etc/containerd/certs.d'
EOF

# Restart containerd
systemctl restart containerd

# Verify it loaded
containerd config dump | grep config_path

----

On the 3 masters

nerdctl handles multi-arch images much better and supports docker login

In [ ]:
wget https://github.com/containerd/nerdctl/releases/download/v2.0.0/nerdctl-2.0.0-linux-amd64.tar.gz
sudo tar Cxzvf /usr/local/bin nerdctl-2.0.0-linux-amd64.tar.gz

nerdctl login -u admin -p 1 master-1:30086

- To get all iamges names and versions

In [ ]:
# Check what images are currently running
kubectl get pods --all-namespaces -o jsonpath='{range .items[*]}{range .spec.containers[*]}{.image}{"\n"}{end}{end}' | sort | uniq

# Check what Kubernetes version you have
kubectl version

# Check kubeadm required images
kubeadm config images list

Only one master node

In [ ]:
cat > cache-all-images.sh << 'EOF'
#!/bin/bash
set -e

NEXUS="master-1:30086"

echo "=========================================="
echo "Pulling from upstream, tagging, pushing to Nexus docker-hosted"
echo "=========================================="

# docker.io images
nerdctl pull docker.io/library/busybox:1.36
nerdctl tag docker.io/library/busybox:1.36 $NEXUS/library/busybox:1.36
nerdctl push --insecure-registry $NEXUS/library/busybox:1.36


nerdctl pull docker.io/clickhouse/clickhouse-server:24.3
nerdctl tag docker.io/clickhouse/clickhouse-server:24.3 $NEXUS/clickhouse/clickhouse-server:24.3
nerdctl push --insecure-registry $NEXUS/clickhouse/clickhouse-server:24.3

nerdctl pull docker.io/sonatype/nexus3:3.64.0
nerdctl tag docker.io/sonatype/nexus3:3.64.0 $NEXUS/sonatype/nexus3:3.64.0
nerdctl push --insecure-registry $NEXUS/sonatype/nexus3:3.64.0

nerdctl pull docker.io/bitnami/kubectl:latest
nerdctl tag docker.io/bitnami/kubectl:latest $NEXUS/bitnami/kubectl:latest
nerdctl push --insecure-registry $NEXUS/bitnami/kubectl:latest

# registry.k8s.io images
for img in kube-apiserver kube-controller-manager kube-scheduler kube-proxy; do
  nerdctl pull registry.k8s.io/${img}:v1.35.5
  nerdctl tag registry.k8s.io/${img}:v1.35.5 $NEXUS/${img}:v1.35.5
  nerdctl push --insecure-registry $NEXUS/${img}:v1.35.5
done

nerdctl pull registry.k8s.io/coredns/coredns:v1.13.1
nerdctl tag registry.k8s.io/coredns/coredns:v1.13.1 $NEXUS/coredns/coredns:v1.13.1
nerdctl push --insecure-registry $NEXUS/coredns/coredns:v1.13.1

nerdctl pull registry.k8s.io/pause:3.10.1
nerdctl tag registry.k8s.io/pause:3.10.1 $NEXUS/pause:3.10.1
nerdctl push --insecure-registry $NEXUS/pause:3.10.1

nerdctl pull registry.k8s.io/etcd:3.6.6-0
nerdctl tag registry.k8s.io/etcd:3.6.6-0 $NEXUS/etcd:3.6.6-0
nerdctl push --insecure-registry $NEXUS/etcd:3.6.6-0

# CSI images
for img in csi-node-driver-registrar:v2.15.0 csi-provisioner:v6.1.0 csi-resizer:v2.0.0 csi-snapshotter:v8.4.0 livenessprobe:v2.17.0 nfsplugin:v4.13.2; do
  name=$(echo $img | cut -d: -f1)
  tag=$(echo $img | cut -d: -f2)
  nerdctl pull registry.k8s.io/sig-storage/${name}:${tag}
  nerdctl tag registry.k8s.io/sig-storage/${name}:${tag} $NEXUS/sig-storage/${name}:${tag}
  nerdctl push --insecure-registry $NEXUS/sig-storage/${name}:${tag}
done

# ghcr.io images
nerdctl pull ghcr.io/flannel-io/flannel:v0.28.4
nerdctl tag ghcr.io/flannel-io/flannel:v0.28.4 $NEXUS/flannel-io/flannel:v0.28.4
nerdctl push --insecure-registry $NEXUS/flannel-io/flannel:v0.28.4

nerdctl pull ghcr.io/headlamp-k8s/headlamp:v0.42.0
nerdctl tag ghcr.io/headlamp-k8s/headlamp:v0.42.0 $NEXUS/headlamp-k8s/headlamp:v0.42.0
nerdctl push --insecure-registry $NEXUS/headlamp-k8s/headlamp:v0.42.0

nerdctl pull ghcr.io/kube-vip/kube-vip:v1.2.0
nerdctl tag ghcr.io/kube-vip/kube-vip:v1.2.0 $NEXUS/kube-vip/kube-vip:v1.2.0
nerdctl push --insecure-registry $NEXUS/kube-vip/kube-vip:v1.2.0

# quay.io images
nerdctl pull quay.io/metallb/controller:v0.15.3
nerdctl tag quay.io/metallb/controller:v0.15.3 $NEXUS/metallb/controller:v0.15.3
nerdctl push --insecure-registry $NEXUS/metallb/controller:v0.15.3

nerdctl pull quay.io/metallb/speaker:v0.15.3
nerdctl tag quay.io/metallb/speaker:v0.15.3 $NEXUS/metallb/speaker:v0.15.3
nerdctl push --insecure-registry $NEXUS/metallb/speaker:v0.15.3

echo "=========================================="
echo "DONE! All images cached in Nexus docker-hosted"
echo "Browse: http://nexus.voip.local → docker-hosted"
echo "=========================================="
EOF

In [ ]:
chmod +x cache-all-images.sh
./cache-all-images.sh

In [ ]:
# Force remove from local cache
# crictl rmi ghcr.io/flannel-io/flannel:v0.28.4 
# nerdctl rmi ghcr.io/flannel-io/flannel:v0.28.4 --force 
sudo ctr images rm docker.io/clickhouse/clickhouse-server:24.3




# Pull using original name — mirror should redirect to Nexus
# nerdctl pull ghcr.io/flannel-io/flannel:v0.28.4

nerdctl pull docker.io/clickhouse/clickhouse-server:24.3

# Check journal to confirm it hit Nexus
journalctl -u containerd --since "2 min ago" | grep -i "30086\|master-1\|mirror\|host"

---

---

Upadted

In [ ]:
cat > nexus-values.yaml <<EOF
persistence:
  enabled: true
  existingClaim: nexus-pvc

tolerations:
- key: "node-role.kubernetes.io/control-plane"
  operator: "Exists"
  effect: "NoSchedule"

nodeSelector:
  kubernetes.io/hostname: master-1

# Fix: Ingress configuration was incorrect
ingress:
  enabled: true
  ingressClassName: nginx
  annotations:
    nginx.ingress.kubernetes.io/proxy-body-size: "0"
    nginx.ingress.kubernetes.io/proxy-read-timeout: "600"
    nginx.ingress.kubernetes.io/proxy-send-timeout: "600"
    nginx.ingress.kubernetes.io/backend-protocol: "HTTP"
  hosts:
    - host: nexus.voip.local
      paths:
        - path: /
          pathType: Prefix

# Fix: Service port configuration
service:
  type: ClusterIP
  port: 8081
  targetPort: 8081

nexus:
  enabled: true
  image:
    repository: sonatype/nexus3
    tag: 3.64.0
  nexusPort: 8081

readinessProbe:
  initialDelaySeconds: 120
  periodSeconds: 15
  failureThreshold: 6
  httpGet:
    path: /service/rest/v1/status
    port: 8081

livenessProbe:
  initialDelaySeconds: 120
  periodSeconds: 15
  failureThreshold: 6
  httpGet:
    path: /service/rest/v1/status
    port: 8081

env:
  - name: INSTALL4J_ADD_VM_PARAMS
    value: "-Xms2703M -Xmx2703M -XX:MaxDirectMemorySize=2703M -XX:+UnlockExperimentalVMOptions -XX:+UseCGroupMemoryLimitForHeap -Djava.util.prefs.userRoot=/nexus-data/javaprefs"
EOF

In [ ]:
# Add critical timeout annotations to the ingress
kubectl annotate ingress -n nexus nexus-nexus-repository-manager \
  nginx.ingress.kubernetes.io/proxy-connect-timeout="600" \
  nginx.ingress.kubernetes.io/proxy-read-timeout="600" \
  nginx.ingress.kubernetes.io/proxy-send-timeout="600" \
  nginx.ingress.kubernetes.io/proxy-next-upstream-timeout="600" \
  nginx.ingress.kubernetes.io/proxy-next-upstream-tries="3" \
  --overwrite

In [ ]:
helm repo add sonatype https://sonatype.github.io/helm3-charts/
helm repo update


In [ ]:
helm install nexus sonatype/nexus-repository-manager \
  -n nexus \
  -f nexus-values.yaml

In [ ]:
helm upgrade nexus sonatype/nexus-repository-manager \
  -n nexus \
  -f nexus-values.yaml

In [ ]:
kubectl logs -n nexus nexus-nexus-repository-manager-846f46776c-7ljs7

In [ ]:
kubectl exec -n nexus -it deployment/nexus-nexus-repository-manager -- cat /nexus-data/admin.password

In [ ]:
curl -v -H "Host: nexus.voip.local" http://172.16.6.90/

---

Test node port ( `Only for test don't use it` )

In [ ]:
# Create a NodePort service for direct access
cat <<EOF | kubectl apply -n nexus -f -
apiVersion: v1
kind: Service
metadata:
  name: nexus-nodeport
spec:
  type: NodePort
  selector:
    app.kubernetes.io/instance: nexus
    app.kubernetes.io/name: nexus-repository-manager
  ports:
  - name: http
    port: 8081
    targetPort: 8081
    nodePort: 30081
EOF

In [ ]:
# Get the IP of any node (use master-1 where Nexus runs)
NODE_IP="172.16.6.66"  # master-1 IP

# Test NodePort access
echo "Testing NodePort access..."
curl -v http://${NODE_IP}:30081/

# If successful, hit it multiple times to warm up
for i in {1..5}; do
  echo "Warm-up request $i..."
  curl -s -o /dev/null -w "HTTP %{http_code} - Time: %{time_total}s\n" http://${NODE_IP}:30081/
  sleep 2
done

# Also hit the status endpoint
curl http://${NODE_IP}:30081/service/rest/v1/status

---

Add repo

In [ ]:
sudo vi /etc/yum.repos.d/nexus.repo


In [ ]:
[nexus-repo]
name=My Nexus Yum Repository
baseurl=http://nexus.voip.local/repository/yum/
enabled=1
gpgcheck=0

Donwload Packges as rpm file

In [ ]:
curl -v --user 'admin:1' --upload-file tzdata-2024b-2.el9.noarch.rpm http://nexus.voip.local/repository/yum/

In [ ]:
curl -v --user 'admin:1' \
     --resolve nexus.voip.local:80:172.16.6.90 \
     --upload-file tzdata-2024b-2.el9.noarch.rpm \
     http://nexus.voip.local/repository/yum/